In [1]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod

In [2]:
random_seed, np_random_seed = random.seed(505), np.random.seed(505)

In [3]:
pd.options.mode.chained_assignment = None  # default='warn'

In [4]:
from modules.classes import Document, Provider, Consumer, Recommender
from modules.samplers import provider_sampler, consumer_sampler
from modules.genre_preferences import generate_genre_preferences

In [5]:
# Movie Lens
ratings_df = pd.read_csv("ml-latest-small/ratings.csv")
movies_df = pd.read_csv("ml-latest-small/movies.csv")
movies_with_ratings_df = ratings_df.merge(movies_df, on="movieId", how="left")
documents_df = (
    movies_with_ratings_df[["movieId", "rating", "genres"]]
    .groupby(["movieId", "genres"])
    .mean()
    .reset_index()
)

In [6]:
movies_with_ratings_df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [7]:
documents_df["rating"] = documents_df["rating"].apply(lambda x: x / 5)

In [8]:
user_genre_preference = generate_genre_preferences(movies_with_ratings_df)

In [9]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [10]:
sampled_users = user_genre_preference.sample(600)
# Shuffle the sampled users to randomize the order
sampled_users = sampled_users.sample(frac=1).reset_index(drop=True)

In [11]:
### Users that are interested in niche movies
ten_percent = int(len(sampled_users) * 0.1)
users_niche_movies = sampled_users[:ten_percent]
users_general_interest = sampled_users[ten_percent:]

In [12]:
# Selecting the Niche Genre
niche_genre = "Western"

other_genres = list(users_niche_movies.columns)
other_genres.remove(niche_genre)

### update the genre preferences for niche movies fans
users_niche_movies[niche_genre] = np.where(
    users_niche_movies[niche_genre] > 0, users_niche_movies[niche_genre] * 4, 0.2
)
users_niche_movies[other_genres] /= 4

### update the genre preferences for the general movie fans
users_general_interest[niche_genre] /= 4

In [13]:
users_niche_movies.columns

Index(['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX',
       'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western'],
      dtype='object')

In [14]:
niche_consumers_list = consumer_sampler(
    category_preferences_df=users_niche_movies, favorite_categories=set([niche_genre])
)
general_consumers_list = consumer_sampler(
    category_preferences_df=users_general_interest
)
consumers_list = (
    niche_consumers_list + general_consumers_list
)  # try it once with niche and another with general
random.shuffle(consumers_list)

In [15]:
len(consumers_list)

600

In [16]:
niche_consumers_set = set([consumer.consumer_id for consumer in niche_consumers_list])
general_consumers_set = set(
    [consumer.consumer_id for consumer in general_consumers_list]
)

In [17]:
# Niche Documents
niche_documents = documents_df[documents_df["genres"].str.contains(niche_genre)]

### SVD Recommender

In [18]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

class MatrixFactorizationRecommender(Recommender):
    def __init__(
        self,
        recommender_id,
        fee_per_click=0,
        fee_per_show=0,
        base_fee=0,
        exploration_prob=0.2,
        specialized_categories=set(),
        prohibited_categories=set(),
        weighted_category={},
        weighted_value=0.5,
    ):
        """
        Initialize the PopularRecommender.

        Args:
            exploration_prob (float): Probability of exploring (randomly selecting documents).
        """
        super().__init__()
        self.recommender_id = recommender_id
        self.exploration_prob = exploration_prob
        self.recommendations = {}
        self.historical_recommendations = []
        self.fee_per_click = fee_per_click
        self.fee_per_show = fee_per_show
        self.base_fee = base_fee
        self.profit = []
        self.specialized_categories = specialized_categories
        self.prohibited_categories = prohibited_categories
        # weighted category variables
        self.weighted_category = (
            weighted_category  # dictionary of category as key and weight as value
        )
        self.weighted_value = weighted_value
        # self.exploration_prob = exploration_prob
        self.algo = SVD()
        self.has_trained_model = False  # Track if the model has been trained yet
        self.precomputed_recommendations = (
            {}
        )  # Dictionary to store precomputed recommendations

    def train_model_if_ready(self):
        """Train the SVD model once there are enough interactions and precompute recommendations."""
        if len(self.interactions) > 10:  # Arbitrary threshold, adjust as needed
            print("Training model")
            reader = Reader(rating_scale=(0, 1))
            data = Dataset.load_from_df(
                pd.DataFrame(
                    self.interactions, columns=["consumer_id", "doc_id", "rating"]
                ),
                reader,
            )
            trainset = data.build_full_trainset()
            self.algo.fit(trainset)
            self.has_trained_model = True
            print("Model training completed")

            # Precompute recommendations for all users
            self.precompute_recommendations()

    def precompute_recommendations(self):
        """Precompute recommendations for all users and store them in a dictionary."""
        print("Precomputing recommendations for all users")
        # Get all unique users and items
        user_ids = {interaction[0] for interaction in self.interactions}
        item_ids = {doc.doc_id for doc in self.documents}

        # Precompute predictions for each user
        for user_id in user_ids:
            predictions = [
                (item_id, self.algo.predict(user_id, item_id).est)
                for item_id in item_ids
            ]
            # Sort items by predicted rating and store the top N items
            sorted_items = sorted(predictions, key=lambda x: x[1], reverse=True)
            self.precomputed_recommendations[user_id] = [
                item_id for item_id, _ in sorted_items
            ]
        print("Precomputation completed")

    def recommend_documents(self, consumers, slate_size=5):
        recommendations = {}

        if not self.has_trained_model:
            self.update_documents_list()  # Collect documents from providers

        for consumer in consumers:
            consumer_id = consumer.consumer_id
            recommended_documents = []

            # Get the list of items already clicked by the consumer
            clicked_items = self.consumer_docs_ids_clicked.get(consumer_id, set())

            # Check if we should explore (cold start) or use precomputed recommendations
            if not self.has_trained_model:
                # Cold start: Recommend random documents excluding already clicked items
                available_documents = [
                    doc for doc in self.documents if doc.doc_id not in clicked_items
                ]
                recommended_documents = random.sample(available_documents, min(slate_size, len(available_documents)))
            else:
                # Retrieve precomputed recommendations, excluding already clicked items
                top_item_ids = [
                    item_id
                    for item_id in self.precomputed_recommendations.get(consumer_id, [])
                    if item_id not in clicked_items
                ][:slate_size]
                recommended_documents = [doc for doc in self.documents if doc.doc_id in top_item_ids]

            # Store recommendations and record shows for tracking purposes
            recommendations[consumer_id] = recommended_documents
            for document in recommended_documents:
                self.record_show(document.provider_id)

        return recommendations

### Single Recommender System

In [19]:
def run_competing_rec_experiment_df_export(
    consumers, providers, recommenders, num_days=5, slate_size=3, num_cycles=1
):
    """
    Run a multi-recommender experiment.

    Args:
        consumers (list): List of Consumer instances.
        providers (list): List of Provider instances.
        recommenders (dict): Dictionary of Recommender instances where keys are recommender IDs.
        num_days (int, optional): Number of days to run the experiment (default is 5).
        slate_size (int, optional): Number of documents to recommend to each consumer (default is 3).

    Returns:
        pd.DataFrame: DataFrame containing provider data.
        pd.DataFrame: DataFrame containing consumer data.
        pd.DataFrame: DataFrame containing recommender system data.
    """
    provider_data = []
    consumer_data = []
    recommender_data = []
    customers_recommender_choice = []

    recommender_values = list(recommenders.values())
    general_recommender = recommender_values[0]

    # Subscribe consumers to the recommender
    for consumer in consumers:
        general_recommender.connect_consumer(consumer)
        consumer.subscribe_to_recommender_system(general_recommender.recommender_id)

    print("General Recommender", general_recommender.connected_consumers.keys())

    # Subscribe providers to both recommenders
    for provider in providers:
        general_recommender.connect_provider(provider)
        provider.subscribe_to_recommender_system(general_recommender.recommender_id)

    ### LOGGING ###
    print(
        "Connected to General recommender:",
        len(general_recommender.connected_consumers),
    )

    # Run the experiment for the specified number of days
    for cycle in range(1, num_cycles + 1):
        for day in range(1, num_days + 1):
            print("===============> Day:", day, "<===============")
            # Organize consumers into lists based on the recommender they chose
            consumers_by_recommender = {
                recommender_id: [] for recommender_id in recommenders.keys()
            }
            for (
                consumer
            ) in consumers:  # Iterate over the consumers for the current recommender ID
                chosen_recommender_id = consumer.choose_recommender()
                if chosen_recommender_id is None:
                    continue
                # Append recommender to consumer to evaluate UCB
                customers_recommender_choice.append(
                    [consumer.consumer_id, chosen_recommender_id]
                )  # used for analysis
                consumers_by_recommender[chosen_recommender_id].append(consumer)

            # Make recommendations for each recommender's associated consumers
            recommendations = {}
            for (
                recommender_id,
                recommender_consumers,
            ) in consumers_by_recommender.items():
                recommender = recommenders[recommender_id]
                slate_documents = recommender.recommend_documents(
                    recommender_consumers, slate_size=slate_size
                )
                for consumer in recommender_consumers:
                    recommendations[consumer.consumer_id] = (
                        recommender_id,
                        slate_documents[consumer.consumer_id],
                    )

            # Simulate user response, update satisfaction scores, and record clicks
            clicked_documents = {
                consumer.consumer_id: [] for consumer in consumers
            }  # Initialize with empty lists
            for consumer in consumers:
                # continue if the consumer is not connected to any recommenders
                if len(consumer.available_recommenders) == 0:
                    print("Consumer is not connected to any recommenders")
                    continue

                consumer_id = consumer.consumer_id
                recommender_id, recommended_documents = recommendations[consumer_id]
                slate_documents = recommended_documents
                chosen_recommender_id = (
                    recommender_id  # Choose recommender for the current user
                )
                responses = consumer.simulate_response(
                    slate_documents, recommender_system_id=chosen_recommender_id
                )

                # Collect clicked items
                for i, response in enumerate(responses):
                    # record interaction (consumer_id, doc_id, rating)
                    recommenders[chosen_recommender_id].add_interaction(
                        consumer_id, slate_documents[i], response["click"]
                    )
                    if response.get("click", 0) == 1:
                        clicked_documents[consumer_id].append(slate_documents[i])
                        recommenders[chosen_recommender_id].record_click(
                            consumer_id, slate_documents[i]
                        )

        # Charge subscription fees to providers
        for recommender in recommenders.values():
            recommender.charge_subscription_fees()

        # Get profit for each provider
        for provider in providers:
            for recommender_id, recommender in recommenders.items():
                if recommender_id in provider.connected_recommenders.keys():
                    provider_data.append(
                        [
                            provider.provider_id,
                            recommender_id,
                            (
                                provider.profit.get(recommender_id, [0])[cycle - 1]
                                if cycle
                                <= len(provider.profit.get(recommender_id, [0]))
                                else 0
                            ),
                            cycle,
                            (
                                provider.pay_cycle_fee.get(recommender_id, [0])[
                                    cycle - 1
                                ]
                                if cycle
                                <= len(provider.pay_cycle_fee.get(recommender_id, [0]))
                                else 0
                            ),
                            (
                                provider.pay_cycle_clicks.get(recommender_id, [0])[
                                    cycle - 1
                                ]
                                if cycle
                                <= len(
                                    provider.pay_cycle_clicks.get(recommender_id, [0])
                                )
                                else 0
                            ),
                            (
                                provider.pay_cycle_shows.get(recommender_id, [0])[
                                    cycle - 1
                                ]
                                if cycle
                                <= len(
                                    provider.pay_cycle_shows.get(recommender_id, [0])
                                )
                                else 0
                            ),
                            len(
                                [
                                    rec
                                    for rec in provider.connected_recommenders.keys()
                                    if provider.connected_recommenders[rec] == 1
                                ]
                            ),
                            (
                                "niche"
                                if provider.provider_id in niche_providers_set
                                else "general"
                            ),
                        ]
                    )
        for consumer in consumers:
            for recommender_id, recommender in recommenders.items():
                if recommender_id in consumer.connected_recommenders.keys():
                    consumer_id = consumer.consumer_id
                    consumer_satisfaction_score = consumer.get_satisfaction_score(
                        recommender_system_id=recommender_id
                    )
                    rec_state = (
                        "connected"
                        if consumer.connected_recommenders[recommender_id] == 1
                        else ""
                    )
                    controlled = (
                        "niche"
                        if consumer.consumer_id in niche_consumers_set
                        else "general"
                    )
                    consumer_data.append(
                        [
                            consumer_id,
                            recommender_id,
                            cycle,
                            consumer_satisfaction_score,
                            rec_state,
                            controlled,
                        ]
                    )

        # Update the subscription for each provider
        for provider in providers:
            unsubscribed_list = provider.update_recommender_subscription()
            for recommender_id, recommender in recommenders.items():
                if recommender_id in unsubscribed_list:
                    recommender.disconnect_provider(provider.provider_id)

        for recommender_id, recommender in recommenders.items():
            num_connected_providers = len(recommender.connected_providers)
            num_connected_consumers = len(recommender.connected_consumers)
            total_profit = recommender.profit[-1]
            recommender_data.append(
                [
                    recommender_id,
                    num_connected_providers,
                    num_connected_consumers,
                    total_profit,
                    cycle,
                ]
            )

        print("\n====================================================")
        print("===============> Finished Cycle:", cycle, "<================")
        print("==================================================== \n")

    # Convert lists to DataFrames
    provider_df = pd.DataFrame(
        provider_data,
        columns=[
            "provider_id",
            "recommender_id",
            "profit",
            "cycle",
            "fee",
            "clicks",
            "shows",
            "connected_recommenders",
            "controlled",
        ],
    )
    consumer_df = pd.DataFrame(
        consumer_data,
        columns=[
            "consumer_id",
            "recommender_id",
            "cycle",
            "satisfaction_score",
            "rec_state",
            "controlled",
        ],
    )
    recommender_df = pd.DataFrame(
        recommender_data,
        columns=[
            "recommender_id",
            "num_connected_providers",
            "num_connected_consumers",
            "total_profit",
            "cycle",
        ],
    )
    customer_recommender_df = pd.DataFrame(
        customers_recommender_choice, columns=["customer_id", "recommender_id"]
    )

    return provider_df, consumer_df, recommender_df, customer_recommender_df

In [20]:
import pandas as pd
import numpy as np
from collections import Counter

# Define the sets of variables for different experiments
experiments = [
    {
        "name": "Experiment 1",
        "num_days": 30,
        "num_cycles": 60,
        "slate_size": 5,
        "num_recommenders": 1,
        "specialized_categories": [
            set(),
        ],
        "prohibited_categories": [
            set(),
        ],
        "weighted_category": [  # value between 0 and 1. default=0.5
            {},
        ],
    },
]

# List of random seeds
random_seeds = [random_seed]

# Initialize DataFrame to store category frequencies per experiment and recommender
category_freq_df = pd.DataFrame(
    columns=["experiment_name", "recommender_id", "category", "frequency"]
)

# Initialize lists to store results of all experiments
all_provider_data = []
all_consumer_data = []
all_recommender_data = []
all_customer_recommender_data = []

# Consumers
num_consumers = consumers_list

# Providers
doc_mean_std = (500, 100)
num_docs = 500
num_providers = 10

# Recommender system base fee
recs_fee = 0
decremental_fee_flag = False  # base_fee = recs_fee / number of recs

for seed in random_seeds:
    np.random.seed(seed)
    random.seed(seed)
    # Loop over each experiment configuration
    for i, exp in enumerate(experiments):
        # Initialize providers
        niche_providers_list = provider_sampler(
            niche_documents,
            int(num_providers * 0.1),
            len(niche_documents),
            starting_id=1,
        )  # 10% are niche providers
        niche_providers_set = set(
            [provider.provider_id for provider in niche_providers_list]
        )

        general_providers_list = provider_sampler(
            documents_df, int(num_providers * 0.9), num_docs, starting_id=2
        )  # 90% are general providers

        providers_list = niche_providers_list + general_providers_list
        random.shuffle(providers_list)

        print(
            f"Running experiment: {exp['name']} | seed: {seed} | number of recommenders: {exp['num_recommenders']}"
        )

        # Initialize recommenders for this experiment
        recommenders = {}
        for j in range(exp["num_recommenders"]):
            recommender_id = f"popular_{j+1}_exp_{i}_seed_{seed}"

            # decrease the fee based on the number of recommenders
            if decremental_fee_flag == True:
                base_fee = recs_fee / exp["num_recommenders"]
            else:
                base_fee = recs_fee

            recommenders[recommender_id] = MatrixFactorizationRecommender(
                recommender_id=recommender_id,
                fee_per_click=0.1,
                fee_per_show=0.01,
                base_fee=base_fee,
                prohibited_categories=exp["prohibited_categories"][
                    j
                ],  # gets the prohibited categories from "prohibited_categories":[[],[]]
                weighted_category=exp["weighted_category"][j],
                specialized_categories=exp["specialized_categories"][j],
            )

        # Run the experiment
        provider_df, consumer_df, recommender_df, customer_recommender_df = (
            run_competing_rec_experiment_df_export(
                consumers=consumers_list,
                providers=providers_list,
                recommenders=recommenders,
                num_days=exp["num_days"],
                slate_size=exp["slate_size"],
                num_cycles=exp["num_cycles"],
            )
        )

        # Calculate category frequencies
        for j in range(exp["num_recommenders"]):
            recommender_id = f"popular_{j+1}_exp_{i}_seed_{seed}"

            category_counter = Counter()
            for doc_list in recommenders[recommender_id].historical_recommendations:
                for doc in doc_list:
                    for category in doc.categories:
                        category_counter[category] += 1

            # Collect rows to be added to category_freq_df
            rows_to_append = []
            for category, frequency in category_counter.items():
                rows_to_append.append(
                    {
                        "experiment_name": exp["name"],
                        "recommender_id": recommender_id,
                        "category": category,
                        "frequency": frequency,
                    }
                )

            # Append rows to DataFrame using concat
            category_freq_df = pd.concat(
                [category_freq_df, pd.DataFrame(rows_to_append)], ignore_index=True
            )

        # Add experiment configuration to the dataframes
        provider_df["experiment_id"] = i
        provider_df["random_seed"] = seed
        consumer_df["experiment_id"] = i
        consumer_df["random_seed"] = seed
        recommender_df["experiment_id"] = i
        recommender_df["random_seed"] = seed
        customer_recommender_df["experiment_id"] = i
        customer_recommender_df["random_seed"] = seed

        # Append the results to the lists
        all_provider_data.append(provider_df)
        all_consumer_data.append(consumer_df)
        all_recommender_data.append(recommender_df)
        all_customer_recommender_data.append(customer_recommender_df)

# Concatenate all results into single dataframes
final_provider_df = pd.concat(all_provider_data, ignore_index=True)
final_consumer_df = pd.concat(all_consumer_data, ignore_index=True)
final_recommender_df = pd.concat(all_recommender_data, ignore_index=True)
final_customer_recommender_df = pd.concat(
    all_customer_recommender_data, ignore_index=True
)

# Save the dataframes to CSV files for analysis
save_dir = "surprise_data_export"
seed = "505"

final_provider_df.to_csv(f"surprise_data_export/provider_data_{seed}.csv", index=False)
final_consumer_df.to_csv(f"surprise_data_export/consumer_data_{seed}.csv", index=False)
final_recommender_df.to_csv(
    f"surprise_data_export/recommender_data_{seed}.csv", index=False
)
final_customer_recommender_df.to_csv(
    f"surprise_data_export/customer_recommender_data_{seed}.csv", index=False
)
category_freq_df.to_csv(
    f"surprise_data_export/category_frequencies_{seed}.csv", index=False
)

print("Experiment Complete!")

Running experiment: Experiment 1 | seed: None | number of recommenders: 1
General Recommender dict_keys([229, 109, 406, 450, 49, 17, 23, 383, 387, 228, 556, 265, 69, 354, 380, 397, 579, 451, 259, 470, 478, 11, 106, 213, 421, 334, 452, 585, 93, 575, 340, 216, 112, 417, 360, 215, 72, 78, 59, 214, 299, 364, 505, 318, 25, 105, 39, 179, 483, 209, 316, 362, 187, 14, 372, 487, 86, 464, 590, 418, 9, 554, 489, 248, 386, 427, 219, 425, 308, 584, 408, 254, 148, 488, 459, 284, 353, 42, 98, 68, 330, 591, 223, 513, 525, 162, 436, 405, 479, 180, 530, 480, 274, 355, 269, 90, 296, 352, 297, 40, 291, 484, 242, 127, 424, 24, 593, 91, 431, 392, 435, 77, 251, 197, 111, 599, 295, 264, 532, 324, 142, 3, 66, 27, 282, 518, 415, 311, 58, 373, 173, 403, 234, 38, 26, 398, 390, 7, 482, 453, 361, 578, 231, 139, 200, 163, 186, 508, 461, 34, 382, 570, 177, 305, 499, 6, 249, 19, 342, 288, 30, 194, 100, 241, 395, 543, 394, 266, 136, 586, 448, 207, 76, 544, 217, 301, 594, 519, 460, 492, 322, 137, 85, 122, 367, 449, 158,

In [21]:
# recommenders["popular_1_exp_0_seed_None"].connected_consumers[0]
conumers_ids = list(
    recommenders["popular_1_exp_0_seed_None"].connected_consumers.keys()
)
for i in conumers_ids:
    consumer_type = "niche" if i in niche_consumers_set else "general"
    print(
        f"{consumer_type} Consumer ID:",
        recommenders["popular_1_exp_0_seed_None"].connected_consumers[i].consumer_id,
    )
    print(
        "Documents Clicked:",
        len(recommenders["popular_1_exp_0_seed_None"].consumer_docs_ids_clicked[i]),
    )
    print()

general Consumer ID: 229
Documents Clicked: 1402

general Consumer ID: 109
Documents Clicked: 351

general Consumer ID: 406
Documents Clicked: 182

general Consumer ID: 450
Documents Clicked: 1043

niche Consumer ID: 49
Documents Clicked: 6

niche Consumer ID: 17
Documents Clicked: 5

niche Consumer ID: 23
Documents Clicked: 8

general Consumer ID: 383
Documents Clicked: 709

general Consumer ID: 387
Documents Clicked: 719

general Consumer ID: 228
Documents Clicked: 693

general Consumer ID: 556
Documents Clicked: 697

general Consumer ID: 265
Documents Clicked: 343

general Consumer ID: 69
Documents Clicked: 220

general Consumer ID: 354
Documents Clicked: 179

general Consumer ID: 380
Documents Clicked: 1021

general Consumer ID: 397
Documents Clicked: 348

general Consumer ID: 579
Documents Clicked: 1090

general Consumer ID: 451
Documents Clicked: 346

general Consumer ID: 259
Documents Clicked: 680

general Consumer ID: 470
Documents Clicked: 174

general Consumer ID: 478
Documen

In [22]:
len(movies_df[movies_df["genres"].str.contains("Western", case=False, na=False)])

167

In [23]:
len(niche_documents)
len(documents_df)

9724